# Minuta: Fine-tuning de mmBERT-small para Detección de Intenciones en Transcripciones

**Startup Minuta** — Alternativa económica a grabadoras con IA (Plaud) basada en celular  
**Proyecto Final Diplomatura en Ciencia de Datos con Python — PUCP**  
**Tarea ODD:** T12 / HU06  

Este notebook ejecuta el fine-tuning supervisado de `jhu-clsp/mmBERT-small` en Google Colab con aceleración GPU para clasificar segmentos transcritos en 5 clases funcionales:
1. `idea`: Temas clave, reflexiones, propuestas metodológicas o teóricas.
2. `tarea`: Compromisos, acciones pendientes (to-dos) y asignaciones con responsable o plazo.
3. `acuerdo`: Decisiones pactadas formalmente en la conversación.
4. `pregunta`: Preguntas abiertas, temas no resueltos o dudas planteadas.
5. `relleno`: Saludos protocolares, cortesías, preámbulos y muletillas orales.

**Modelo Base:** `jhu-clsp/mmBERT-small`  
**Destino en Hugging Face Hub:** `rodnm/minuta-detector-mmbert-small`  
**Oráculo de Validación (Hipótesis H4):** F1 Macro ≥ 0.70 y Recall de `tarea` ≥ 0.80.

In [ ]:
# 1. Verificación del entorno GPU (CUDA)
!nvidia-smi
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo activo: {device}')
if torch.cuda.is_available():
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}')

In [ ]:
# 2. Instalación de dependencias en Google Colab
!pip install -q transformers datasets evaluate scikit-learn accelerate huggingface_hub

## 1. Carga y Aplanamiento del Dataset Sintético

Cargamos `sinteticos_entrenamiento.json`, generado con la API de Gemini (98.45% de segmentos únicos). Aplanamos las conversaciones a una lista estructurada de segmentos con texto y etiqueta funcional.

In [ ]:
import json
import os
from pathlib import Path

CLASES = ['idea', 'tarea', 'acuerdo', 'pregunta', 'relleno']
LABEL2ID = {c: i for i, c in enumerate(CLASES)}
ID2LABEL = {i: c for i, c in enumerate(CLASES)}

# Localizar archivo sinteticos_entrenamiento.json
posibles_rutas = [
    Path('entrenamiento/datos/sinteticos_entrenamiento.json'),
    Path('sinteticos_entrenamiento.json'),
    Path('../entrenamiento/datos/sinteticos_entrenamiento.json'),
]
ruta_datos = None
for r in posibles_rutas:
    if r.exists():
        ruta_datos = r
        break

if ruta_datos is None:
    print('Clonando repositorio para acceder a los datos de entrenamiento...')
    !git clone -q https://github.com/rodnorabuena/startup-project-diplomado-pucp.git repo
    ruta_datos = Path('repo/entrenamiento/datos/sinteticos_entrenamiento.json')

print(f'Cargando dataset desde: {ruta_datos}')
raw_data = json.loads(ruta_datos.read_text(encoding='utf-8'))

ejemplos = []
for conv in raw_data.get('conversaciones', []):
    for seg in conv.get('segmentos', []):
        txt = seg.get('texto', '').strip()
        lbl = seg.get('etiqueta', '')
        if txt and lbl in CLASES:
            ejemplos.append({'texto': txt, 'etiqueta': lbl})

print(f'Total de segmentos útiles cargados: {len(ejemplos)}')

## 2. Análisis de Desbalance y Cálculo de Pesos de Clase

Dado que clases críticas para Minuta como `tarea` y `acuerdo` son minoritarias frente a `idea`, aplicamos la fórmula estándar de ponderación para `CrossEntropyLoss`:
$$w_c = \frac{N}{C \cdot n_c}$$
donde $N$ es el total de muestras, $C$ el número de clases y $n_c$ la frecuencia de la clase $c$.

In [ ]:
from collections import Counter
import numpy as np

etiquetas = [e['etiqueta'] for e in ejemplos]
conteos = Counter(etiquetas)
total = len(etiquetas)
num_clases = len(CLASES)

print('Distribución de frecuencias por clase:')
pesos_dict = {}
for c in CLASES:
    n = conteos[c]
    pct = (n / total) * 100
    w = total / (num_clases * n)
    pesos_dict[c] = float(w)
    print(f'  - {c:10s}: {n:4d} muestras ({pct:5.2f}%) -> Peso asignado: {w:.4f}')

tensor_pesos = torch.tensor([pesos_dict[ID2LABEL[i]] for i in range(num_clases)], dtype=torch.float)

## 3. Tokenización y División Estratificada (80/20)

Cargamos el tokenizador de `jhu-clsp/mmBERT-small` con longitud máxima truncada a 128 tokens y dividimos estratificadamente.

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import datasets

MODELO_BASE = 'jhu-clsp/mmBERT-small'
tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

train_data, val_data = train_test_split(
    ejemplos,
    test_size=0.20,
    random_state=42,
    stratify=[e['etiqueta'] for e in ejemplos]
)

print(f'Muestras para Entrenamiento: {len(train_data)}')
print(f'Muestras para Validación:    {len(val_data)}')

def tokenizar_lote(batch):
    enc = tokenizer(batch['texto'], padding='max_length', truncation=True, max_length=128)
    enc['labels'] = [LABEL2ID[lbl] for lbl in batch['etiqueta']]
    return enc

ds_train = datasets.Dataset.from_list(train_data).map(tokenizar_lote, batched=True)
ds_val = datasets.Dataset.from_list(val_data).map(tokenizar_lote, batched=True)

## 4. Trainer con Función de Pérdida Ponderada y Métricas HDD

Definimos `WeightedTrainer` para inyectar los pesos de clase en `CrossEntropyLoss` y configuramos `compute_metrics` para calcular Accuracy, F1 Macro y el Recall específico de `tarea` (H4).

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch.nn as nn
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro', zero_division=0)
    
    id_tarea = LABEL2ID['tarea']
    recalls = recall_score(labels, preds, average=None, zero_division=0)
    recall_tarea = float(recalls[id_tarea]) if id_tarea < len(recalls) else 0.0
    
    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'recall_tarea': recall_tarea,
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        weights = tensor_pesos.to(logits.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## 5. Entrenamiento del Modelo mmBERT-small

Configuramos hiperparámetros con optimizador AdamW, tasa de aprendizaje $3 \times 10^{-5}$ y evaluación por época.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE,
    num_labels=len(CLASES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

training_args = TrainingArguments(
    output_dir='./checkpoints_mmbert',
    num_train_epochs=4,
    per_device_train_batch_size=16 if torch.cuda.is_available() else 4,
    per_device_eval_batch_size=32 if torch.cuda.is_available() else 8,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    seed=42,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    compute_metrics=compute_metrics,
)

print('Iniciando fine-tuning supervisado...')
train_output = trainer.train()
print('Entrenamiento finalizado con éxito.')

## 6. Evaluación Detallada y Verificación del Oráculo H4

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

eval_res = trainer.evaluate()
print('=' * 60)
print('MÉTRICAS EN CONJUNTO DE VALIDACIÓN:')
print(f"  Accuracy:        {eval_res['eval_accuracy']:.4f}")
print(f"  F1 Macro:        {eval_res['eval_f1_macro']:.4f}  (Umbral H4: >= 0.70)")
print(f"  Recall de Tarea: {eval_res['eval_recall_tarea']:.4f}  (Umbral H4: >= 0.80)")
print('=' * 60)

pred_output = trainer.predict(ds_val)
y_pred = np.argmax(pred_output.predictions, axis=-1)
y_true = pred_output.label_ids

print('\nReporte de Clasificación por Clase:')
print(classification_report(y_true, y_pred, target_names=CLASES, digits=4))

print('Matriz de Confusión:')
print(confusion_matrix(y_true, y_pred))

## 7. Pruebas de Inferencia de Ejemplo con Jerga ES-EN

In [ ]:
from transformers import pipeline

clasificador = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

frases = [
    'Debemos redactar la response letter para el peer review antes del deadline fijado.',
    'La arquitectura de mmBERT-small permite inferencia rápida en CPU para audios largos.',
    'Quedamos en priorizar la postulación al grant de Concytec este semestre.',
    '¿Cuál es el monto estimado del APC para publicar en open access en este journal?',
    'Buenos días con todos los docentes e investigadores que nos acompañan.',
]

print('Predicciones del modelo afinado:')
for frase in frases:
    r = clasificador(frase)[0]
    print(f"Texto:  '{frase}'")
    print(f"Pred:   {r['label']} (confianza: {r['score']:.4f})\n")

## 8. Guardado Local y Publicación en Hugging Face Hub

In [ ]:
salida_local = 'modelos/minuta-detector-mmbert-small'
model.save_pretrained(salida_local)
tokenizer.save_pretrained(salida_local)
print(f'Modelo y tokenizer guardados localmente en: {salida_local}')

HUB_MODEL_ID = 'rodnm/minuta-detector-mmbert-small'
print(f'Target de publicación: {HUB_MODEL_ID}')

# Para publicar en el Hub, ingrese su token con permisos Write:
# from huggingface_hub import login
# login()
# model.push_to_hub(HUB_MODEL_ID)
# tokenizer.push_to_hub(HUB_MODEL_ID)